## 7. Summary / 요약

**English**: We illustrated the core quantitative content of Arlt & Vaquero (2020):
- **Wolf vs. Group SN** differ systematically; $G_s$ is more robust for historical observers.
- **Synthetic 400-year series** reproduces the MM (~4 groups/yr) vs. modern max (100+/yr) ratio.
- **k-factor correction** recovers a weak observer to within ~1% of the reference series.
- **Butterfly diagram** from Spörer's law encodes the cycle signature back to 1610.
- **Wavelet analysis** recovers Schwabe, Gleissberg, and Suess periodicities.
- **Uncertainty envelopes** grow dramatically in grand minima, matching the data-sparsity situation discussed in the review.

These quantitative tools mirror those used by digitization teams (Arlt 2008, Arlt et al. 2013, 2016; Karoff et al. 2019) to reduce pre-photographic drawings to modern machine-readable datasets.

**Korean**: 본 노트북은 Arlt & Vaquero (2020)의 핵심 정량적 내용을 예시했다:
- **Wolf vs. 군 SN**은 계통적으로 다르며; $G_s$는 역사 관측자에 더 강건.
- **400년 합성 계열**이 MM(~4 군/년) vs. 현대 최대(100+/년) 비율을 재현.
- **k-factor 보정**이 약한 관측자를 참조 계열의 ~1% 오차 이내로 복원.
- Spörer 법칙 기반 **butterfly diagram**이 1610년까지의 주기 signature를 인코딩.
- **Wavelet 분석**이 Schwabe, Gleissberg, Suess 주기를 복원.
- **불확정 envelope**이 대극소기에 극적으로 확장되며, 리뷰에서 논의된 자료 희소 상황과 일치.

이 정량 도구는 디지털화 팀(Arlt 2008, Arlt et al. 2013, 2016; Karoff et al. 2019)이 사진 이전 스케치를 현대적 기계판독 데이터셋으로 환원하는 데 쓴 기법을 반영한다.

In [ ]:
def bootstrap_envelope(series: np.ndarray, k_std: np.ndarray,
                        n_boot: int = 300) -> tuple:
    """Bootstrap a 5-95% envelope by perturbing with observer-k uncertainty.

    Args:
        series: Annual SSN mean.
        k_std: Per-year k-factor perturbation standard deviation.
        n_boot: Number of bootstrap samples.

    Returns:
        Tuple (lower, upper) 5th and 95th percentile envelopes.
    """
    draws = np.empty((n_boot, len(series)))
    for i in range(n_boot):
        k = rng.normal(1.0, k_std)
        draws[i] = np.maximum(0, k * series + rng.normal(0, 2 + 0.05 * series))
    return np.percentile(draws, 5, axis=0), np.percentile(draws, 95, axis=0)


# Mimic MM and Dalton: larger k-std in those epochs (more observer ambiguity)
k_std_arr = np.where(((years_ann >= 1645) & (years_ann <= 1715)) |
                     ((years_ann >= 1790) & (years_ann <= 1830)),
                     0.40, 0.12)
lower, upper = bootstrap_envelope(ssn_ann, k_std_arr)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(years_ann, lower, upper, color="steelblue", alpha=0.25,
                label="5-95% bootstrap envelope")
ax.plot(years_ann, ssn_ann, color="k", lw=0.9, label="Annual mean")
ax.axvspan(1645, 1715, alpha=0.12, color="red", label="Maunder Min.")
ax.axvspan(1790, 1830, alpha=0.12, color="orange", label="Dalton Min.")
ax.set_xlabel("Year")
ax.set_ylabel("Annual SSN")
ax.set_title("Reconstruction uncertainty envelope / 재구성 불확정 envelope")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

mm_mask_ann = (years_ann >= 1645) & (years_ann <= 1715)
mod_mask_ann = (years_ann >= 1950) & (years_ann <= 2000)
print(f"Mean envelope width (MM):      {(upper - lower)[mm_mask_ann].mean():.1f}")
print(f"Mean envelope width (modern):  {(upper - lower)[mod_mask_ann].mean():.1f}")

## 6. Reconstruction Uncertainty Envelope / 재구성 불확정 envelope

**English**: Historical SSN reconstructions have large uncertainties. We use bootstrap resampling to estimate a 5–95% envelope around the annual mean. The envelope widens dramatically during the Maunder Minimum and Dalton Minimum — matching the data-sparsity situation described by Arlt & Vaquero.

**Korean**: 역사적 SSN 재구성은 큰 불확정성을 갖는다. Bootstrap 재표집으로 연간 평균 주위의 5–95% envelope를 추정한다. Envelope는 Maunder Minimum과 Dalton Minimum에서 극적으로 넓어진다 — Arlt & Vaquero가 기술한 자료 희소 상황과 일치한다.

In [ ]:
def morlet_cwt(signal: np.ndarray, dt: float, periods: np.ndarray,
               w0: float = 6.0) -> np.ndarray:
    """Simple Morlet continuous wavelet transform (amplitude only).

    Args:
        signal: 1D time series.
        dt: Sampling interval (years).
        periods: Array of analysis periods (years).
        w0: Morlet dimensionless angular frequency (default 6).

    Returns:
        2D array |W(period, t)| of wavelet amplitudes.
    """
    n = len(signal)
    scales = periods * w0 / (2 * np.pi)
    W = np.zeros((len(periods), n))
    for i, s in enumerate(scales):
        window = max(int(6 * s / dt), 3)
        offsets = np.arange(-window, window + 1) * dt
        psi = (np.pi ** -0.25) * np.exp(1j * w0 * offsets / s) * \
              np.exp(-0.5 * (offsets / s) ** 2)
        re = np.convolve(signal, np.real(psi[::-1]), mode="same") / np.sqrt(s)
        im = np.convolve(signal, np.imag(psi[::-1]), mode="same") / np.sqrt(s)
        W[i] = np.sqrt(re ** 2 + im ** 2)
    return W


# Downsample synthetic series to annual for speed
years_ann = np.arange(1610, 2020, 1)
ssn_ann = np.array([ssn_syn[(years >= y) & (years < y + 1)].mean() for y in years_ann])

periods = np.logspace(np.log10(5), np.log10(300), 40)
W = morlet_cwt(ssn_ann - ssn_ann.mean(), dt=1.0, periods=periods)

fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                       gridspec_kw={"height_ratios": [1, 2]})
ax[0].plot(years_ann, ssn_ann, color="C0", lw=0.8)
ax[0].set_ylabel("Annual SSN")
ax[0].set_title("Annual synthetic SSN and its wavelet power / 연간 SSN과 wavelet 파워")

X, Y = np.meshgrid(years_ann, periods)
im = ax[1].contourf(X, Y, W ** 2, levels=25, cmap="viridis")
ax[1].set_yscale("log")
ax[1].set_ylabel("Period (yr)")
ax[1].set_xlabel("Year")
ax[1].axhline(11, color="white", ls="--", lw=0.7, alpha=0.8)
ax[1].axhline(88, color="orange", ls="--", lw=0.7, alpha=0.8)
ax[1].axhline(210, color="red", ls="--", lw=0.7, alpha=0.8)
ax[1].text(2025, 11, "Schwabe", color="white", va="center", fontsize=8)
ax[1].text(2025, 88, "Gleissberg", color="orange", va="center", fontsize=8)
ax[1].text(2025, 210, "Suess", color="red", va="center", fontsize=8)
plt.colorbar(im, ax=ax[1], label="Power")
plt.tight_layout()
plt.show()

## 5. Wavelet Analysis of Long-term SSN / 장기 SSN의 Wavelet 분석

**English**: We use a Morlet-like continuous wavelet transform to reveal the Schwabe (~11-yr), Gleissberg (~88-yr), and Suess/de Vries (~210-yr) periodicities in our synthetic series. For a real-data analogue of the Hallstatt (~2300-yr) cycle we would need cosmogenic-isotope records (Wu et al. 2018).

**Korean**: Morlet형 연속 wavelet 변환으로 합성 계열에서 Schwabe(~11년), Gleissberg(~88년), Suess/de Vries(~210년) 주기를 드러낸다. Hallstatt(~2300년)의 실제 자료 대응물은 우주 기원 동위원소 기록(Wu et al. 2018)이 필요하다.

In [ ]:
def spoerer_latitude(phase: np.ndarray, start_lat: float = 28.0,
                      end_lat: float = 6.0) -> np.ndarray:
    """Mean sunspot emergence latitude as a function of cycle phase (Spörer's law).

    Args:
        phase: Cycle phase in [0, 1].
        start_lat: Latitude at cycle start (degrees).
        end_lat: Latitude at cycle end.

    Returns:
        Mean |latitude| in degrees.
    """
    return start_lat - (start_lat - end_lat) * np.clip(phase, 0, 1)


def sample_butterfly(years_grid: np.ndarray,
                     n_spots_per_year: np.ndarray) -> tuple:
    """Sample spot latitudes following Spörer's law with hemispheric split.

    Args:
        years_grid: Array of years (assumed 1/12-year spaced).
        n_spots_per_year: Expected spot count at each time.

    Returns:
        Tuple (times, latitudes).
    """
    T_cycle = 11.0
    all_t, all_lat = [], []
    for tt, nn in zip(years_grid, n_spots_per_year):
        if nn < 0.1:
            continue
        n_sample = rng.poisson(nn)
        if n_sample == 0:
            continue
        phase_i = ((tt - 1755.0) / T_cycle) % 1.0
        mean_lat = spoerer_latitude(np.full(n_sample, phase_i))
        hemi = rng.choice([-1, 1], size=n_sample)
        lats = hemi * (mean_lat + rng.normal(0, 5, size=n_sample))
        all_t.append(np.full(n_sample, tt))
        all_lat.append(lats)
    if not all_t:
        return np.array([]), np.array([])
    return np.concatenate(all_t), np.concatenate(all_lat)


scale = ssn_syn / 5  # proxy for number of spots to sample
t_spots, lat_spots = sample_butterfly(years, scale)

fig, ax = plt.subplots(figsize=(13, 5))
colors = np.where((t_spots >= 1645) & (t_spots <= 1715), "red", "steelblue")
ax.scatter(t_spots, lat_spots, s=0.3, c=colors, alpha=0.4)
ax.axhline(0, color="k", lw=0.5)
ax.axvspan(1645, 1715, alpha=0.08, color="red")
ax.axvspan(1790, 1830, alpha=0.08, color="orange")
ax.set_ylim(-45, 45)
ax.set_xlim(1610, 2020)
ax.set_xlabel("Year")
ax.set_ylabel("Heliographic latitude (deg)")
ax.set_title("Synthetic butterfly diagram 1610-2020 / 1610-2020 합성 butterfly diagram")
plt.tight_layout()
plt.show()

## 4. Synthetic Butterfly Diagram from Historical Sketches / 역사 스케치로부터의 butterfly diagram

**English**: Using Spörer's law (mean latitude drifts from ~±28° at cycle start to ~±6° at cycle end), we synthesize a butterfly that mimics Arlt & Vaquero's Fig. 27 (cycles 0–24). The Maunder Minimum appears as a barren window.

**Korean**: Spörer 법칙(주기 시작 ~±28°에서 끝 ~±6°로 평균 위도 이동)을 써서 Arlt & Vaquero의 그림 27(주기 0–24)을 모방하는 butterfly를 합성한다. Maunder Minimum은 빈 창(窓)으로 나타난다.

In [ ]:
def calibrate_k(series_obs: np.ndarray, series_ref: np.ndarray) -> float:
    """Estimate an observer's k-factor by ratio of means in a common epoch.

    Args:
        series_obs: Observer's SSN series for the common-epoch window.
        series_ref: Reference SSN series for the same window.

    Returns:
        Estimated k-hat factor (k_obs * k_hat -> k_ref scale).
    """
    mask = (series_ref > 1) & (series_obs > 1)
    if not mask.any():
        return 1.0
    return series_ref[mask].mean() / series_obs[mask].mean()


common_mask = (years >= 1950) & (years <= 2000)
ssn_ref = ssn_syn  # treat the synthetic as ground truth
k_true = 0.65
ssn_weak = k_true * ssn_syn + rng.normal(0, 4, size=ssn_syn.shape)
ssn_weak = np.maximum(0, ssn_weak)

k_hat = calibrate_k(ssn_weak[common_mask], ssn_ref[common_mask])
ssn_weak_corrected = k_hat * ssn_weak

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(years, ssn_ref, lw=0.6, color="k", label="Reference (k=1)")
ax[0].plot(years, ssn_weak, lw=0.6, color="C3", alpha=0.7,
           label=f"Raw observer (k_true={k_true})")
ax[0].set_ylabel("SSN")
ax[0].set_title("Before k-factor correction / k-factor 보정 전")
ax[0].legend()

ax[1].plot(years, ssn_ref, lw=0.6, color="k", label="Reference")
ax[1].plot(years, ssn_weak_corrected, lw=0.6, color="C2", alpha=0.7,
           label=f"Corrected (k_hat={k_hat:.3f})")
ax[1].set_ylabel("SSN")
ax[1].set_xlabel("Year")
ax[1].set_title("After k-factor correction / k-factor 보정 후")
ax[1].legend()
plt.tight_layout()
plt.show()

print(f"True k:          {k_true}")
print(f"Estimated k-hat: {k_hat:.3f}  (expected ~{1/k_true:.3f})")
print(f"Relative error:  {100*(k_hat - 1/k_true)/(1/k_true):.2f}%")

## 3. Observer Completeness (k-factor) Correction / 관측자 완전성 보정

**English**: Different observers have systematically different count levels. We simulate a "weak telescope" observer who misses small spots ($k_\text{true}=0.65$) and use the common-epoch ratio to recover $\hat{k}$, then correct the series. This is essentially the Wolf-Wolfer calibration procedure.

**Korean**: 관측자별로 계통적인 계수 레벨 차이가 있다. 작은 흑점을 놓치는 "약한 망원경" 관측자($k_\text{참}=0.65$)를 시뮬레이션하고 공통 시기 비율로 $\hat{k}$을 복원하여 계열을 보정한다. 이는 본질적으로 Wolf-Wolfer 보정 절차다.

In [ ]:
def waldmeier_shape(phase: np.ndarray, rise_fraction: float = 0.35) -> np.ndarray:
    """Asymmetric Waldmeier-style solar cycle shape.

    Args:
        phase: Cycle phase in [0, 1] where 0 = minimum, 1 = next minimum.
        rise_fraction: Fraction of cycle length spent in rising phase.

    Returns:
        Normalized amplitude in [0, 1].
    """
    p = np.clip(phase, 0, 1)
    out = np.zeros_like(p)
    rise = p < rise_fraction
    decay = ~rise
    out[rise] = np.sin(0.5 * np.pi * p[rise] / rise_fraction) ** 2
    tail = (p[decay] - rise_fraction) / (1 - rise_fraction)
    out[decay] = np.exp(-3.0 * tail) * np.sin(0.5 * np.pi * (1 - tail)) ** 2 * 4
    return out / out.max() if out.max() > 0 else out


def maunder_suppression(year: np.ndarray,
                         mm_start: float = 1645,
                         mm_end: float = 1715,
                         dalton_start: float = 1790,
                         dalton_end: float = 1830) -> np.ndarray:
    """Envelope that suppresses cycle amplitudes in grand minima.

    Args:
        year: Array of years.
        mm_start: Start year of Maunder Minimum.
        mm_end: End year of Maunder Minimum.
        dalton_start: Start year of Dalton Minimum.
        dalton_end: End year of Dalton Minimum.

    Returns:
        Suppression factor in [0, 1] (1 = normal, ~0.05 = deep minimum).
    """
    env = np.ones_like(year, dtype=float)
    mm_mid = 0.5 * (mm_start + mm_end)
    mm_width = (mm_end - mm_start) / 2
    env *= 1 - 0.95 * np.exp(-((year - mm_mid) / (mm_width * 0.6)) ** 2)
    dalton_mid = 0.5 * (dalton_start + dalton_end)
    dalton_width = (dalton_end - dalton_start) / 2
    env *= 1 - 0.55 * np.exp(-((year - dalton_mid) / (dalton_width * 0.7)) ** 2)
    return env


years = np.arange(1610, 2020, 1 / 12)
T_cycle = 11.0
phase = ((years - 1755.0) / T_cycle) % 1.0
base_shape = waldmeier_shape(phase)

gleissberg = 1 + 0.25 * np.sin(2 * np.pi * (years - 1900) / 88)
suess = 1 + 0.15 * np.sin(2 * np.pi * (years - 1750) / 210)
noise = rng.normal(0, 0.08, size=years.shape)
envelope = 130 * gleissberg * suess * maunder_suppression(years) * (1 + noise)
ssn_syn = np.maximum(0, base_shape * envelope)
g_annual_syn = ssn_syn / 15  # convert R to approximate group count

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(years, ssn_syn, lw=0.7, color="C0", label="Synthetic monthly SSN")
ax.axvspan(1645, 1715, alpha=0.15, color="red", label="Maunder Minimum 1645-1715")
ax.axvspan(1790, 1830, alpha=0.15, color="orange", label="Dalton Minimum 1790-1830")
ax.set_xlabel("Year")
ax.set_ylabel("Synthetic SSN")
ax.set_title("400-year synthetic sunspot number with grand minima / 대극소기 포함 400년 합성 흑점수")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

mm_mask = (years >= 1645) & (years <= 1715)
modern_mask = (years >= 1950) & (years <= 2000)
print(f"MM annual mean group count:     {g_annual_syn[mm_mask].mean():.2f} groups/yr (paper: ~4/yr)")
print(f"Modern annual mean group count: {g_annual_syn[modern_mask].mean():.1f} groups/yr (paper: 100+)")

## 2. Synthetic 400-year Cycle Series with Maunder Minimum / Maunder 극소기를 포함한 400년 합성 주기

**English**: We build a Waldmeier-shape cycle (fast rise, slow decay) and modulate its amplitude with three superposed modes: Gleissberg (~88 yr), Suess/de Vries (~210 yr), plus a Maunder-Minimum suppression window (1645–1715) and Dalton (1790–1830). This mirrors what the historical record shows: annual group counts of order ~4 during the MM vs. 100+ during modern maxima, as discussed by Arlt & Vaquero.

**Korean**: 빠른 상승·느린 하강의 Waldmeier 모양 주기를 만들고 세 모드를 중첩하여 진폭을 변조한다: Gleissberg(~88년), Suess/de Vries(~210년), 그리고 Maunder Minimum 억제 창(1645–1715)과 Dalton(1790–1830). 이는 역사 기록과 일치한다: MM 동안 연간 군수 ~4 vs. 현대 최대기 100+, Arlt & Vaquero가 논의한 바와 같다.

In [ ]:
def wolf_sn(groups: np.ndarray, spots: np.ndarray, k: float = 1.0) -> np.ndarray:
    """Compute Wolf relative sunspot number R = k (10 g + n).

    Args:
        groups: Array of group counts g.
        spots: Array of total individual spot counts n.
        k: Observer completeness/scaling factor (Zurich reference = 1.0).

    Returns:
        Array of Wolf relative sunspot numbers R.
    """
    return k * (10.0 * groups + spots)


def group_sn(groups_by_observer: np.ndarray, k_factors: np.ndarray) -> float:
    """Compute Hoyt & Schatten (1998) group sunspot number for a single day.

    Args:
        groups_by_observer: 1D array of group counts g_i per observer.
        k_factors: 1D array of observer-specific k'_i factors.

    Returns:
        Daily group sunspot number G_s.
    """
    N = len(groups_by_observer)
    if N == 0:
        return 0.0
    return (12.08 / N) * np.sum(k_factors * groups_by_observer)


# Simulate 200 years of monthly data with a ~11-year cycle + Gleissberg envelope
t = np.arange(1820, 2020, 1 / 12)
phase = 2 * np.pi * (t - 1823.3) / 11.0
amplitude_envelope = 80 + 40 * np.sin(2 * np.pi * (t - 1900) / 100)
g_true = np.maximum(0, 0.5 * amplitude_envelope * (0.5 - 0.5 * np.cos(phase)))
n_true = 10 * g_true + rng.normal(0, 3, size=g_true.shape)
n_true = np.maximum(0, n_true)

R_zurich = wolf_sn(g_true, n_true, k=1.0)
Gs_zurich = 12.08 * g_true  # single-observer G_s approximation

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(t, R_zurich, lw=0.8, color="C0", label="Wolf SN R")
ax[0].set_ylabel("Wolf R")
ax[0].legend(loc="upper right")
ax[0].set_title("Synthetic monthly sunspot indices 1820-2020 / 1820-2020 합성 월별 흑점 지수")
ax[1].plot(t, Gs_zurich, lw=0.8, color="C3", label="Group SN $G_s$")
ax[1].set_ylabel("Group SN")
ax[1].set_xlabel("Year")
ax[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

mask = (t >= 1900) & (t < 2000)
print(f"Mean Wolf R (1900-2000):    {R_zurich[mask].mean():.1f}")
print(f"Mean Group SN (1900-2000):  {Gs_zurich[mask].mean():.1f}")
print(f"Ratio R/G_s:                {R_zurich[mask].mean() / Gs_zurich[mask].mean():.3f}")

## 1. Wolf SN vs. Group SN Reconstruction / Wolf vs. 군 흑점수 비교

**English**: Wolf's number $R = k(10g + n)$ is sensitive to *individual spot counts* — a difference that matters enormously for small-telescope historical observers who often missed pores. The Group Sunspot Number $G_s = (12.08/N)\sum k'_i g_i$ (Hoyt & Schatten 1998) uses only group counts and is therefore more robust to observer differences. Here we illustrate the two indices side by side on a synthetic record.

**Korean**: Wolf의 수 $R = k(10g + n)$은 **개별 흑점 수**에 민감하다 — 작은 망원경을 쓴 역사 관측자들은 기공(pore)을 놓치기 쉬워 이 차이는 막대하다. 군 흑점수 $G_s = (12.08/N)\sum k'_i g_i$ (Hoyt & Schatten 1998)는 군 수만 쓰므로 관측자 차이에 강건하다. 여기서 두 지수를 합성 기록에서 나란히 보인다.

In [ ]:
"""Imports and global styling for historical sunspot analysis."""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams["figure.dpi"] = 110
rcParams["axes.grid"] = True
rcParams["grid.alpha"] = 0.3
rcParams["font.size"] = 10

rng = np.random.default_rng(seed=42)
print("NumPy:", np.__version__)

# Historical Sunspot Records — Implementation / 역사적 흑점 기록 구현

**Paper**: Arlt, R. & Vaquero, J. M. (2020). *Historical sunspot records*. Living Reviews in Solar Physics **17**, 1. DOI: 10.1007/s41116-020-0023-y

## Overview / 개요

**English**: This notebook reproduces the quantitative skeleton of the Arlt & Vaquero (2020) review. We (i) compare Wolf vs. Group sunspot number formulas, (ii) build a synthetic 400-year Waldmeier-shape cycle series with a realistic Maunder-Minimum suppression, (iii) illustrate observer completeness (k-factor) correction, (iv) construct a synthetic butterfly diagram from idealized Spörer's law, (v) perform wavelet analysis revealing Gleissberg and Suess/de Vries periods, and (vi) quantify reconstruction uncertainty envelopes with bootstrap.

**Korean**: 본 노트북은 Arlt & Vaquero (2020) 리뷰의 정량적 골격을 재현한다. (i) Wolf vs. 군 흑점수 공식 비교, (ii) Maunder Minimum 억제가 반영된 400년 합성 Waldmeier형 주기 계열, (iii) 관측자 완전성 (k-factor) 보정 예시, (iv) Spörer 법칙 기반 합성 butterfly diagram, (v) Gleissberg·Suess/de Vries 주기를 드러내는 wavelet 분석, (vi) bootstrap 기반 재구성 불확정 envelope를 다룬다.